# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Classification, clustering, ranking, or scoring — which one, and why?

In [ ]:
"""
This is a ranking/scoring task. I want to rank pages by how likely they are to need a refresh 
so an editor can review the most important ones first.

"""

In [ ]:
import os, pandas as pd

candidates = [
  "../../data/raw/content_refresh_anonymized.csv",
  "data/raw/content_refresh_anonymized.csv",
]
path = next((p for p in candidates if os.path.exists(p)), None)
df = pd.read_csv(path, low_memory=False)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [4]:
# I would predict whether a page should be prioritized for refresh.
# I will use a proxy label based on the observed trend signal in the data.
# The label is `trend_direction == "down"`.

# Sketch what the target column would look like

df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
df[['content_id', 'trend_direction', 'is_declining_label']].head()


,content_id,trend_direction,is_declining_label
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# The output is a ranked queue, not a binary alert system.
# Editors review a fixed number of pages per week — editorial capacity is the constraint.
# What matters is: are the pages at the top of the queue actually declining?
#
# Average Precision (AP) measures exactly this:
# it rewards a model that ranks true declining pages higher,
# and penalizes one that buries them behind stable pages.
#
# F2 would be appropriate if we were making a binary decision with no ranking.
# Here we are ranking — AP is the correct metric.
#
# A secondary threshold metric: Precision at K (P@50)
# meaning: of the top 50 pages we send to editors, how many are truly declining?
# This maps directly to editorial capacity and is easy to explain to stakeholders.

from sklearn.metrics import average_precision_score
import numpy as np

# Illustrative example: model scores vs true labels
np.random.seed(42)
true_labels = np.array([1]*50 + [0]*50)
model_scores = np.random.uniform(0, 1, 100)
model_scores[:50] += 0.3  # declining pages score slightly higher

ap = average_precision_score(true_labels, model_scores)
print(f"Average Precision (illustrative): {ap:.3f}")

# Precision at K = 50
k = 50
top_k_indices = np.argsort(model_scores)[::-1][:k]
precision_at_k = true_labels[top_k_indices].mean()
print(f"Precision at K=50 (illustrative): {precision_at_k:.3f}")

print("\nTarget: AP > 0.70 and P@50 > 0.60")
print("These numbers mean 'good' because they reflect real editorial value,")
print("not just statistical accuracy on a balanced dataset.")


0.815068493150685

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# df already loaded above (section 2 setup) with a portable relative path
# One row = one content page (content item), with aggregated 90-day performance and metadata.
df[['content_id', 'client_id', 'content_type', 'impressions_90d', 'sessions_90d', 'ctr', 'trend_direction']].head()


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [11]:

# A simple rule is too weak because many signals matter at once.
# Traffic, freshness, position, engagement, and content type all matter.
# ML can learn combinations of those signals better than a hand-written rule.

df[['impressions_90d', 'sessions_90d', 'ctr', 'content_age_days', 'days_since_last_update', 'trend_direction']].describe(include='all')


,impressions_90d,sessions_90d,ctr,content_age_days,days_since_last_update,trend_direction
count,30000.000000,30000.000000,30000.000000,30000.00000,30000.000000,30000
unique,NaN,NaN,NaN,NaN,NaN,5
top,NaN,NaN,NaN,NaN,NaN,down
freq,NaN,NaN,NaN,NaN,NaN,16262
mean,5200.366300,37.066633,0.510733,256.16780,46.098300,NaN
std,16838.019547,107.069131,3.279162,132.70793,42.078709,NaN
min,1.000000,1.000000,0.000000,90.00000,1.000000,NaN
25%,81.000000,2.000000,0.000000,132.00000,20.000000,NaN
50%,731.000000,7.000000,0.070000,236.00000,20.000000,NaN
75%,3615.250000,27.000000,0.290000,333.00000,104.000000,NaN


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.